### 为链怎加记忆能力
---
* 注意：简单的链的记忆添加可以使用v0.2的方式，复杂的官方推荐使用langgraph
* 短时记忆：InMemoryHistory
* 长时记忆： RunnableWithMessageHistory

#### InMemoryHistory

In [8]:
from typing import List # 导入List类型提示
from pydantic import BaseModel, Field  # 导入Pydantic的BaseModel和Field
from langchain_core.chat_history import BaseChatMessageHistory  # 导入聊天历史基类

from langchain_core.messages import BaseMessage, AIMessage  # 导入消息基类和AI消息类

class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    """内存中实现的聊天消息历史记录"""

    messages: List[BaseMessage] = Field(default_factory=list)  # 使用空列表作为默认值存储消息

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """添加一组消息到存储中"""
        self.messages.extend(messages)

    def clear(self) -> None:
        """清空所有消息"""
        self.messages = []

# 这里我们使用全局变量来存储聊天消息历史。
# 这样可以更容易地检查它，以查看底层结果。
store = {}  # 创建空字典用于存储不同会话的历史记录

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    """根据会话ID获取历史记录，如果不存在则创建新的"""
    if session_id not in store:
        store[session_id] = InMemoryHistory()  # 为新会话创建新的历史记录对象
    return store[session_id]

# 获取会话ID为"1"的历史记录
history = get_by_session_id("1")
# 添加一个条AI消息到历史记录
history.add_message(AIMessage(content="你好")) # 修改为中文消息
# 打印存储的所有历史记录
print(store)  # 将输出包含会话"1"的历史记录，其中有一条“你好”的AI消息

{'1': InMemoryHistory(messages=[AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}


在链中增加短时记忆

In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # 导入聊天模板和消息占位符
from langchain_core.runnables.history import RunnableWithMessageHistory  # 导入带历史记录的可运行组件
from langchain_core.chat_history import BaseChatMessageHistory  # 导入聊天历史基类
from langchain_core.messages import BaseMessage  # 导入消息基类和AI消息类
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field  # 导入Pydantic的BaseModel和Field
from dotenv import load_dotenv
from typing import List # 导入List类型提示
import os

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    """内存中实现的聊天消息历史记录"""

    messages: List[BaseMessage] = Field(default_factory=list)  # 使用空列表作为默认值存储消息

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """添加一组消息到存储中"""
        self.messages.extend(messages)

    def clear(self) -> None:
        """清空所有消息"""
        self.messages = []

store = {}

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    """根据会话ID获取历史记录，如果不存在则创建新的"""
    if session_id not in store:
        store[session_id] = InMemoryHistory()  # 为新会话创建新的历史记录对象
    return store[session_id]


# 创建聊天提示模板，包含系统提示、历史记录和用户问题
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个擅长{ability}的助手"), # 系统角色提示，使用ability变量定义助手专长
    MessagesPlaceholder(variable_name="history"), # 放置历史消息的占位符
    ("human", "{question}"),  # 用户问题的占位符
])

# 将提示模板与DS模型连接成一个链
chain = prompt | model

# 创建带有消息历史功能的可运行链
chain_with_history= RunnableWithMessageHistory(
    chain,  # 基础链
    get_by_session_id,
    input_messages_key="question", # 指定输入消息的键名
    history_messages_key="history" # 指定历史消息的键名
)

# 首次调用链，询问余弦的含义
print(chain_with_history.invoke(
    {"ability":"math", "question":"余弦函数是什么意思？"}, # 输入参数
    config={"configurable":{"session_id":"foo"}}  # 配置会话ID为“foo”
))

# 打印存储中的历史记录
# 此时应包含第一次对话的问题和回答
print(store)

c:\Users\Lxb\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


content='余弦函数是三角函数的一种，通常记作 **cos**。\n\n### 1. 直角三角形中的定义\n在一个直角三角形中，取一个锐角 \\(\\theta\\)，则\n\n\\[\n\\cos\\theta=\\frac{\\text{邻边}}{\\text{斜边}}\n\\]\n\n其中“邻边”是这个角旁边的一条直角边，“斜边”是最长的那条边。\n\n例如：若角 \\(\\theta\\) 的邻边长为 3，斜边长为 5，则\n\n\\[\n\\cos\\theta=\\frac{3}{5}=0.6\n\\]\n\n### 2. 单位圆中的定义\n在平面直角坐标系中画一个半径为 1 的圆（单位圆）。从原点出发作一条与 \\(x\\) 轴正方向夹角为 \\(\\theta\\) 的射线，与单位圆交于点 \\(P(x,y)\\)，则\n\n\\[\n\\cos\\theta=x\n\\]\n\n也就是说，余弦值等于该点的横坐标。\n\n### 3. 图像和性质\n余弦函数\n\n\\[\ny=\\cos x\n\\]\n\n的图像是一条周期性的波浪线：\n\n- 定义域：所有实数\n- 值域：\\([-1,1]\\)\n- 周期：\\(2\\pi\\)\n- 是偶函数，即 \\(\\cos(-x)=\\cos x\\)\n- 特殊值：  \n  \\[\n  \\cos 0=1,\\quad \\cos\\frac{\\pi}{2}=0,\\quad \\cos\\pi=-1\n  \\]\n\n### 4. 和正弦的关系\n余弦和正弦关系密切：\n\n\\[\n\\cos\\theta=\\sin\\left(\\theta+\\frac{\\pi}{2}\\right)\n\\]\n\n也可以理解为：余弦描述的是“角度变化时，横向坐标如何变化”。\n\n### 简单总结\n**余弦函数就是用来描述一个角的“邻边与斜边之比”的函数**；在单位圆中，它表示角度对应点的横坐标。它广泛用于几何、物理、工程、信号处理等领域。' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户问“余弦函数是什么意思？”需要解释。可能是数学定义。要简洁清晰。可以讲直角三角形定义、单位圆定义、图像

In [ ]:
# 第二次调用链，询问余弦的反函数
# 由于使用相同的会话ID，模型可以参考前一次对话的上下文
print(chain_with_history.invoke(
    {"ability":"math", "question": "它的反函数是什么？"}, # 输入参数
    config={"configurable":{"session_id": "foo"}} # 使用相同的会话ID
))

# 再次打印存储中的历史记录
# 此时应包含两次对话的完整历史
print(store)

content='你还没有给出具体的函数表达式，所以无法确定“它的反函数”。\n\n请把函数写出来，例如：\n- \\(y=2x+3\\)\n- \\(y=x^2\\)\n- \\(y=e^x\\)\n\n我就能帮你求反函数，并说明定义域和值域限制。' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户问“它的反函数是什么？”但没给“它”指什么。需要澄清。也许上下文缺失。应说明需要具体函数。可以给一般定义。中文简洁。询问提供函数表达式。若指某函数如？ 需要问。最终应请求补充。'} response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 41, 'total_tokens': 167, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 63, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 41}, 'model_provider': 'deepseek', 'model_name': 'deepseek-ai/DeepSeek-V4.1-Flash', 'system_fingerprint': '', 'id': '01a1a9a1919e46de684a5b612231ceb5', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1a9a1-8ac1-7663-be46-2ee95501a2f7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_

In [11]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState

# 1. 创建一个内存检查点保存器（自动管理 session）
memory = MemorySaver()

# 2. 定义一个简单的图（Graph）
builder = StateGraph(MessagesState)

def call_model(state: MessagesState):
    # 直接把历史消息 + 新消息传给模型
    response = model.invoke(state["messages"])
    return {"messages": [response]}

builder.add_node("model", call_model)
builder.set_entry_point("model")

# 3. 编译图，并传入 memory
graph = builder.compile(checkpointer=memory)

# 4. 调用时，通过 config 指定 thread_id (相当于之前的 session_id)
config = {"configurable": {"thread_id": "foo"}}

# 第一次对话
graph.invoke({"messages": [("human", "余弦函数是什么意思？")]}, config)

# 第二次对话 (模型会自动从 memory 中读取 thread_id="foo" 的历史)
graph.invoke({"messages": [("human", "那它的导数是什么？")]}, config)

{'messages': [HumanMessage(content='余弦函数是什么意思？', additional_kwargs={}, response_metadata={}, id='3d50351b-f1db-4721-b24f-b1ce0a1f547f'),
  AIMessage(content='余弦函数是三角函数的一种，通常记作 **cos**。\n\n它可以这样理解：\n\n1. **在直角三角形中**  \n   对于一个锐角 \\( \\theta \\)：\n   \\[\n   \\cos\\theta=\\frac{\\text{邻边}}{\\text{斜边}}\n   \\]\n   也就是：角旁边的那条直角边，除以斜边。\n\n2. **在单位圆中**  \n   在半径为 1 的圆上，从正 x 轴开始转角度 \\( \\theta \\)，对应点的横坐标就是：\n   \\[\n   \\cos\\theta\n   \\]\n   所以余弦函数可以看成“转一个角度后，横向位置是多少”。\n\n3. **常见值**\n   \\[\n   \\cos 0^\\circ=1\n   \\]\n   \\[\n   \\cos 90^\\circ=0\n   \\]\n   \\[\n   \\cos 180^\\circ=-1\n   \\]\n\n4. **图像和性质**\n   - 余弦函数的图像是一条周期性波浪线。\n   - 周期是 \\(360^\\circ\\)，也就是 \\(2\\pi\\) 弧度。\n   - 它是偶函数，满足：\n     \\[\n     \\cos(-\\theta)=\\cos\\theta\n     \\]\n   - 最大值是 1，最小值是 -1。\n\n5. **有什么用？**\n   余弦函数常用于描述周期性现象和几何关系，比如：\n   - 三角形边长计算\n   - 声音、光、交流电的波动\n   - 物理中的力分解\n   - 计算机图形学、信号处理等\n\n简单说：**余弦函数就是“给定一个角度，求出这个角度对应的邻边与斜边的比值”的函数。**', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要

增加对话ID，精准控制记忆

In [13]:
from langchain_deepseek import ChatDeepSeek
from langchain_core.runnables import (
    ConfigurableFieldSpec,
)

from dotenv import load_dotenv
import os

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

store = {}  # 创建空字典用于存储不同用户和对话的历史记录

def get_session_history(
        user_id: str, conversation_id: str
) -> BaseChatMessageHistory:
    """
    根据用户ID和对话ID获取聊天历史记录
    如果不存在则创建新的历史记录对象

    参数：
        user_id: 用户的唯一标识符
        conversation_id: 对话的唯一标识符

    返回：
        对应的聊天历史记录对象
    """
    if (user_id, conversation_id) not in store:
        store[(user_id, conversation_id)] = InMemoryHistory()
    return store[(user_id, conversation_id)]

# 创建聊天提示模板，包含系统提示，历史记录和用户问题
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个擅长{ability}的助手"), # 系统角色提示，使用ability变量定义助手专长
    MessagesPlaceholder(variable_name="history"), # 放置历史消息的占位符
    ("human", "{question}"),  # 用户问题的占位符
])

# 将提示模板与DS模型连接成一个链
chain = prompt | model

# 创建带有消息历史功能的可运行链，支持用户ID和对话ID配置
with_message_history = RunnableWithMessageHistory(
    chain, # 基础链
    get_session_history=get_session_history,  # 获取历史记录的函数
    input_messages_key="question", # 输入消息的键名
    history_messages_key="history", # 历史消息的键名
    history_factory_config=[  # 历史记录工厂配置
        ConfigurableFieldSpec(
            id="user_id",  # 配置字段ID
            annotation=str, # 类型注解
            name="用户ID",  # 字段名称
            description="用户的唯一标识符",  # 字段描述
            default="", # 默认值
            is_shared=True, # 是否在多个调用间共享
        ),
        ConfigurableFieldSpec(
            id="conversation_id",  # 配置字段ID
            annotation=str, # 类型注解
            name="对话ID",  # 字段名称
            description="对话的唯一标识符",  # 字段描述
            default="", # 默认值
            is_shared=True, # 是否在多个调用间共享
        ),
    ],
)


# 调用链，询问余弦的含义
# 指定用户ID为“123”，对话ID为“1”
with_message_history.invoke(
    {"ability":"数学", "question": "余弦函数是什么意思？"}, # 输入参数
    config={"configurable": {"user_id":"123", "conversation_id": "1"}}  # 配置参数
)


AIMessage(content='余弦函数（cosine function，记作 \\(\\cos\\)）是三角函数的一种，用来描述角度与边长比值或单位圆上点的横坐标之间的关系。\n\n### 1. 直角三角形中的定义\n在一个直角三角形中，对于一个锐角 \\(\\theta\\)：\n\n\\[\n\\cos \\theta=\\frac{\\text{邻边}}{\\text{斜边}}\n\\]\n\n也就是说，余弦值等于这个角的邻边长度除以斜边长度。\n\n例如：\n- \\(\\cos 0^\\circ=1\\)\n- \\(\\cos 60^\\circ=\\frac12\\)\n- \\(\\cos 90^\\circ=0\\)\n\n### 2. 单位圆中的定义\n在平面直角坐标系中，以原点为圆心作单位圆。从 \\(x\\) 轴正方向开始转一个角 \\(\\theta\\)，终边与单位圆交于点 \\((x,y)\\)，则：\n\n\\[\n\\cos \\theta=x\n\\]\n\n即余弦函数表示单位圆上对应点的横坐标。\n\n### 3. 函数性质\n余弦函数 \\(y=\\cos x\\) 有一些重要性质：\n- 定义域：全体实数\n- 值域：\\([-1,1]\\)\n- 周期：\\(2\\pi\\)\n- 是偶函数：\\(\\cos(-x)=\\cos x\\)\n- 图像是一条周期性波浪线\n\n### 4. 简单理解\n正弦和余弦都可以描述“角度对应的比例关系”。其中：\n- 正弦 \\(\\sin\\theta\\) 常对应单位圆上的纵坐标；\n- 余弦 \\(\\cos\\theta\\) 常对应单位圆上的横坐标。\n\n余弦函数在几何、物理、工程、信号处理等领域都有广泛应用，例如描述波动、振动、交流电和周期运动。', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户问“余弦函数是什么意思？”需要解释。要简洁清楚。可以介绍定义：直角三角形锐角邻边/斜边，单位圆横坐标，函数图像周期偶函数，公式。也可以提到与正弦关系。避免过度。需要适合。可以给例子 cos0=1 cos90=0。解释“余弦”英文 cosine。可能问数学意义。最终中文。要覆盖：几

In [15]:
print(store)  # 输出存储的历史记录

{('123', '1'): InMemoryHistory(messages=[HumanMessage(content='余弦函数是什么意思？', additional_kwargs={}, response_metadata={}), AIMessage(content='余弦函数（cosine function，记作 \\(\\cos\\)）是三角函数的一种，用来描述角度与边长比值或单位圆上点的横坐标之间的关系。\n\n### 1. 直角三角形中的定义\n在一个直角三角形中，对于一个锐角 \\(\\theta\\)：\n\n\\[\n\\cos \\theta=\\frac{\\text{邻边}}{\\text{斜边}}\n\\]\n\n也就是说，余弦值等于这个角的邻边长度除以斜边长度。\n\n例如：\n- \\(\\cos 0^\\circ=1\\)\n- \\(\\cos 60^\\circ=\\frac12\\)\n- \\(\\cos 90^\\circ=0\\)\n\n### 2. 单位圆中的定义\n在平面直角坐标系中，以原点为圆心作单位圆。从 \\(x\\) 轴正方向开始转一个角 \\(\\theta\\)，终边与单位圆交于点 \\((x,y)\\)，则：\n\n\\[\n\\cos \\theta=x\n\\]\n\n即余弦函数表示单位圆上对应点的横坐标。\n\n### 3. 函数性质\n余弦函数 \\(y=\\cos x\\) 有一些重要性质：\n- 定义域：全体实数\n- 值域：\\([-1,1]\\)\n- 周期：\\(2\\pi\\)\n- 是偶函数：\\(\\cos(-x)=\\cos x\\)\n- 图像是一条周期性波浪线\n\n### 4. 简单理解\n正弦和余弦都可以描述“角度对应的比例关系”。其中：\n- 正弦 \\(\\sin\\theta\\) 常对应单位圆上的纵坐标；\n- 余弦 \\(\\cos\\theta\\) 常对应单位圆上的横坐标。\n\n余弦函数在几何、物理、工程、信号处理等领域都有广泛应用，例如描述波动、振动、交流电和周期运动。', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户问“余弦函数是什么意思？”需要解释。要

### 使用Redis构建长期记忆
---
* 安装redis 
* 运行redis服务
* 配置长期记忆


启动redis服务器：

In [18]:
! redis-stack-server

'redis-stack-server' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [17]:
! pip install -qU langchain-redis langchain-openai redis

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.4.7 requires langchain-core<2.0.0,>=1.6.13, but you have langchain-core 0.3.86 which is incompatible.
langchain-anthropic 1.7.7 requires langchain-core<2.0.0,>=1.6.10, but you have langchain-core 0.3.86 which is incompatible.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.3.86 which is incompatible.
langchain-chroma 1.1.0 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-community 0.4.2 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.3.86 which is 

In [24]:
! pip install langchain-community redis

   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 580.5/580.5 kB 2.0 MB/s  0:00:00
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.7 MB 6.4 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/12.7 MB 12.8 MB/s eta 0:00:01
   ------------------------------- -------- 10.0/12.7 MB 18.2 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 18.5 MB/s  0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 1.26.4

    Uninstalling numpy-1.26.4:

      Successfully uninstalled numpy-1.26.4

   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-deepseek 1.1.1 requires langchain-openai<2.0.0,>=1.3.1, but you have langchain-openai 0.3.35 which is incompatible.
langchain-openai 0.3.35 requires langchain-core<1.0.0,>=0.3.78, but you have langchain-core 1.7.6 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


测试redis连接正常

In [29]:
import os

# Use the environment variable if set, otherwise default to localhost
REDIS_URL = "redis://localhost:6379"
print(f"Connnecting to Redis at:{REDIS_URL}")


# 依赖包
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import AIMessage, HumanMessage
from  langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

# 最简单的使用
# 初始化 Redis 聊天消息历史记录
# 使用Redis存储聊天历史，需要提供会话 ID 和 Redis 连接 URL
history = RedisChatMessageHistory(session_id="user_123", url=REDIS_URL)
history.clear()  # 首先清空历史记录
# 向历史记录中添加消息
history.add_user_message("你好，AI助手！") # 添加用户消息
history.add_ai_message("你好！有什么可以帮助你的？")  # 添加AI回复消息

# 检索并显示历史消息
print("聊天历史：")
for message in history.messages:
    # 打印每条消息的内容和类型
    print(f"{type(message).__name__}: {message.content}")



Connnecting to Redis at:redis://localhost:6379
聊天历史：
HumanMessage: 你好，AI助手！
AIMessage: 你好！有什么可以帮助你的？
